# BF16 vs FP32 Precision, and the Default Rounding Mode (PyTorch Native Beta 5)

This notebook explores **BF16 precision** relative to FP32 on AWS Trainium, and uses it to
determine -- and empirically confirm on device -- the **default stochastic-rounding (SR) behavior**
of a **PyTorch Native Beta 5** environment (SDK 2.32-era: `torch 2.12.1`, `neuronx-cc 2.27.2878`,
`nki 0.6.0`).

## Background

BF16 has an 8-bit significand (7 stored mantissa bits), so near a value of magnitude `2^e` the
spacing between representable BF16 numbers -- the **ULP** -- is `2^(e-7)`. An arbitrary FP32 number
almost never lands exactly on the BF16 grid; it falls between two neighbors and must be **rounded**.

Trainium's TensorEngine accumulates matmuls in **FP32** (in PSUM), then downcasts the result to
BF16 on write-out. That downcast uses one of two modes:

- **RNE** (round-nearest-even) -- deterministic. A given FP32 value always maps to the *same* BF16
  neighbor (whichever is closer; ties go to even).
- **SR** (stochastic rounding) -- the value rounds *up* with probability equal to its fractional
  distance to the upper neighbor, else *down*. Any single result is random, but the **expected
  value is unbiased** (equals the FP32 input on average), which helps training accumulate small updates.

SR is toggled by the runtime env var **`NEURON_RT_STOCHASTIC_ROUNDING_EN`**, read once at `nrt_init()`.

**We test 5 FP32 values spanning five orders of magnitude**, so you can see the BF16 ULP grow with
magnitude, and watch each rounding mode behave:

- **RNE section:** each FP32 value produces exactly **one** BF16 result.
- **Stochastic section:** each FP32 value produces a **range** of BF16 results (its two bracketing
  grid points), and their sample mean tracks the FP32 value more closely than the single RNE value.

Because the env var is latched at process start, each configuration runs in its **own subprocess**.

## 1. Environment check

In [1]:
import os
os.environ.setdefault("NEURON_RT_LOG_LEVEL", "ERROR")  # hide benign trn1 sync-IO warnings
import os, sys, subprocess, textwrap
import torch, torch_neuronx  # noqa

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
try:
    import nki; print("nki    :", getattr(nki, "__version__", "?"))
except Exception as e:
    print("nki    : (n/a)", e)
print()
print(subprocess.run(["neuron-ls"], capture_output=True, text=True).stdout)

On-device RNG not available - CPU RNG will be used


python : 3.12.11
torch  : 2.12.1+cu130
nki    : 0.6.0+30289107548.gd2d9cc57

instance-type: trn1.2xlarge
instance-id: i-07ae7d44cfca4fb0f
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 2      | 0-1      | 32 GB  | 0000:00:1e.0 | 0-7      | -1   |
+--------+--------+----------+--------+--------------+----------+------+



## 2. The five test values, and their BF16 neighbors (CPU reference)

Before touching the device, we compute -- with `ml_dtypes` on the CPU -- what BF16 *should* do:
each value's ULP, the two BF16 grid points that bracket it, and the fractional position between them
(which is exactly the probability SR will round up).

In [2]:
import math, numpy as np

VALUES = [0.10000001, 1.00390625, 3.1415927, 100.7, 12345.678]

def bf16_neighbors(v):
    f = float(np.float32(v))
    e = math.floor(math.log2(abs(f)))
    ulp = 2.0 ** (e - 7)                 # bf16 spacing at this magnitude
    lower = math.floor(f / ulp) * ulp
    upper = lower + ulp
    frac_up = (f - lower) / ulp          # == P(round up) under SR
    return f, ulp, lower, upper, frac_up

try:
    import ml_dtypes
    def rne(v):
        return float(np.array([np.float32(v)]).astype(ml_dtypes.bfloat16).astype(np.float32)[0])
except ImportError:
    ml_dtypes = None
    def rne(v):
        f, ulp, lo, hi, fr = bf16_neighbors(v); return hi if fr > 0.5 else lo

print(f"{'fp32':>13} {'bf16 ULP':>11} {'lower':>13} {'upper':>13} {'frac_up':>8} {'RNE':>13} {'rel_err':>9}")
for v in VALUES:
    f, ulp, lo, hi, fr = bf16_neighbors(v)
    r = rne(v); rel = abs(r - f) / abs(f)
    print(f"{f:13.7g} {ulp:11.3e} {lo:13.7g} {hi:13.7g} {fr:8.3f} {r:13.7g} {rel:9.2e}")

         fp32    bf16 ULP         lower         upper  frac_up           RNE   rel_err
          0.1   4.883e-04    0.09960938     0.1000977    0.800     0.1000977  9.76e-04
     1.003906   7.812e-03             1      1.007812    0.500             1  3.89e-03
     3.141593   1.562e-02      3.140625       3.15625    0.062      3.140625  3.08e-04
        100.7   5.000e-01         100.5           101    0.400         100.5  1.99e-03
     12345.68   6.400e+01         12288         12352    0.901         12352  5.12e-04


Notice the ULP climbs from ~4.9e-4 (near 0.1) to 64 (near 12345) -- BF16 keeps ~2-3 significant
decimal digits regardless of magnitude, so the *absolute* gap between representable numbers grows
with the value. `frac_up` is where each value sits between its neighbors; it will predict the
stochastic split in Section 4.

## 3. RNE section (default): one BF16 result per FP32 value

We run the **default** environment (`NEURON_RT_STOCHASTIC_ROUNDING_EN` unset) on device. Each FP32
value is pushed through a matmul FP32 accumulator `N` times and downcast to BF16. To feed an
arbitrary FP32 value into a BF16-input matmul without pre-rounding it, we split it into two BF16
partials (`hi + residual`); the K=2 accumulator reconstructs the full FP32 value in PSUM, so only
the *final* accumulator->BF16 write-out is rounded -- the step the SR flag controls.

Expectation: **exactly one distinct BF16 result** per value (deterministic).

In [3]:
WORKER = textwrap.dedent(r"""
    import os, torch, torch_neuronx  # noqa
    dev = "privateuseone:0"
    VALUES = [0.10000001, 1.00390625, 3.1415927, 100.7, 12345.678]
    N = 8192

    def decomp(v):
        f = torch.tensor([v], dtype=torch.float32)
        hi = f.to(torch.bfloat16).to(torch.float32)
        res = (f - hi).to(torch.bfloat16).to(torch.float32)
        return float(hi.item()), float(res.item())

    def round_on_device(v):
        hi, res = decomp(v)
        a = torch.zeros((N, 2), dtype=torch.bfloat16)
        a[:, 0] = hi; a[:, 1] = res
        a = a.to(dev)
        b = torch.ones((2, 1), dtype=torch.bfloat16).to(dev)
        return (a @ b).squeeze(-1).to(torch.float32).cpu()

    en = os.environ.get("NEURON_RT_STOCHASTIC_ROUNDING_EN")
    print("NEURON_RT_STOCHASTIC_ROUNDING_EN =", en)
    for v in VALUES:
        out = round_on_device(v)
        vals, counts = torch.unique(out, return_counts=True)
        vlist = [round(x, 7) for x in vals.tolist()]
        probs = [round(c / out.numel(), 3) for c in counts.tolist()]
        mean = float(out.mean())
        fp = float(torch.tensor([v], dtype=torch.float32).item())
        print(f"fp32={fp:12.7g} | distinct={vals.numel()} results={vlist} probs={probs} "
              f"| mean={mean:.7g} mean_relerr={abs(mean-fp)/abs(fp):.2e}")
""")

def run_device(en):
    env = dict(os.environ)
    if en is None: env.pop("NEURON_RT_STOCHASTIC_ROUNDING_EN", None)
    else:          env["NEURON_RT_STOCHASTIC_ROUNDING_EN"] = en
    env["NEURON_RT_MAP_HBM"] = "0"       # trn1 compatibility
    env["NEURON_RT_LOG_LEVEL"] = "ERROR" # hide benign trn1 sync-IO warnings
    env["NEURON_LAUNCH_BLOCKING"] = "1"  # synchronous error reporting
    r = subprocess.run([sys.executable, "-c", WORKER], env=env, capture_output=True, text=True)
    out = r.stdout.strip()
    return out if out else ("NO OUTPUT\n" + r.stderr[-1500:])

print(">>> RNE (default, NEURON_RT_STOCHASTIC_ROUNDING_EN unset):\n")
print(run_device(None))

>>> RNE (default, NEURON_RT_STOCHASTIC_ROUNDING_EN unset):



NEURON_RT_STOCHASTIC_ROUNDING_EN = None
fp32=         0.1 | distinct=1 results=[0.1000977] probs=[1.0] | mean=0.1000977 mean_relerr=9.76e-04
fp32=    1.003906 | distinct=1 results=[1.0] probs=[1.0] | mean=1 mean_relerr=3.89e-03
fp32=    3.141593 | distinct=1 results=[3.140625] probs=[1.0] | mean=3.140625 mean_relerr=3.08e-04
fp32=       100.7 | distinct=1 results=[100.5] probs=[1.0] | mean=100.5 mean_relerr=1.99e-03
fp32=    12345.68 | distinct=1 results=[12352.0] probs=[1.0] | mean=12352 mean_relerr=5.12e-04


Every value collapses to a **single** BF16 number (`distinct=1`), and the sample mean equals that
same number -- deterministic RNE. The relative error is bounded by half a ULP: tiny for values that
happen to sit near a grid point, larger for values near the midpoint (e.g. `1.00390625`, an exact tie).

## 4. Stochastic section (`=1`): a range of BF16 results per FP32 value

Now we set `NEURON_RT_STOCHASTIC_ROUNDING_EN=1`. Each FP32 value should now round to **both** of its
bracketing BF16 neighbors across the `N` trials, split according to `frac_up` from Section 2 -- and
the **sample mean** should track the FP32 value far better than the single RNE value did.

In [4]:
print(">>> Stochastic (NEURON_RT_STOCHASTIC_ROUNDING_EN=1):\n")
print(run_device("1"))

>>> Stochastic (NEURON_RT_STOCHASTIC_ROUNDING_EN=1):



NEURON_RT_STOCHASTIC_ROUNDING_EN = 1
fp32=         0.1 | distinct=2 results=[0.0996094, 0.1000977] probs=[0.198, 0.802] | mean=0.100001 mean_relerr=9.69e-06
fp32=    1.003906 | distinct=2 results=[1.0, 1.0078125] probs=[0.49, 0.51] | mean=1.003981 mean_relerr=7.41e-05
fp32=    3.141593 | distinct=2 results=[3.140625, 3.15625] probs=[0.939, 0.061] | mean=3.141584 mean_relerr=2.66e-06
fp32=       100.7 | distinct=2 results=[100.5, 101.0] probs=[0.603, 0.397] | mean=100.6986 mean_relerr=1.38e-05
fp32=    12345.68 | distinct=2 results=[12288.0, 12352.0] probs=[0.103, 0.897] | mean=12345.44 mean_relerr=1.95e-05


Compare the two runs value-by-value:

- **RNE**: `distinct=1`, mean = the one rounded value. Rounding error is *systematic* (always the
  same direction for a given input).
- **SR**: `distinct=2` (the two BF16 neighbors), `probs` match `frac_up`, and **`mean_relerr` is much
  smaller** -- often by one to three orders of magnitude -- because the random ups and downs average
  out to the true FP32 value. That unbiasedness is exactly why SR helps training accumulate many
  small BF16 updates without drift, at the cost of per-sample determinism.

## 5. Summary

| | RNE (default) | SR (`NEURON_RT_STOCHASTIC_ROUNDING_EN=1`) |
|---|---|---|
| distinct BF16 results per FP32 value | **1** | **2** (the bracketing grid points) |
| deterministic? | yes | no (seedable via `NEURON_RT_STOCHASTIC_ROUNDING_SEED`) |
| per-sample error | up to half a ULP, systematic | up to one ULP, zero-mean |
| sample mean vs FP32 | = the rounded value | tracks FP32 (unbiased) |

**Conclusions:**

1. **BF16 precision is magnitude-relative**: ULP `= 2^(e-7)`, so absolute rounding error grows with
   the value's magnitude while relative error stays ~`2^-8`.
2. **The default rounding mode in Beta 5 is RNE -- stochastic rounding is OFF by default.** Each FP32
   value produces a single deterministic BF16 result unless SR is explicitly enabled.
3. **SR trades per-sample determinism for an unbiased mean**, which is why it targets training, not
   inference. It acts on the matmul PSUM FP32->BF16 downcast path (set `NEURON_RT_STOCHASTIC_ROUNDING_EN=1`).